<a href="https://colab.research.google.com/github/SenTier1107/Appprogramming_2026/blob/main/0417_FastAPI_Prerequisites/gradio_crud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
실습과제 1 — Gradio CRUD Dashboard
customers.csv 기반 | 고객ID 자동 배정 | Pydantic 유효성 검사 | Gradio Blocks
"""

# ── 설치 (Colab 환경) ──
# !pip install gradio pydantic pandas --quiet

import sqlite3
import os
import re
import pandas as pd
import gradio as gr
from pydantic import BaseModel, field_validator, ValidationError

# 기존 세션 종료 + DB 초기화
gr.close_all()
if os.path.exists("customer_gradio.db"):
    os.remove("customer_gradio.db")

# ════════════════════════════════════════
# 1. 설정
# ════════════════════════════════════════
DB_NAME = "customer_gradio.db"
CSV_URL = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"


# ════════════════════════════════════════
# 2. Pydantic 모델
# ════════════════════════════════════════
class CustomerInput(BaseModel):
    """Create/Update 입력값 검증 — 텍스트 항목에 숫자 불가"""
    성별: str
    결제수단: str
    거주지: str
    회원등급: str
    만족도: int
    선호온도: float
    나이: int

    @field_validator('성별', '결제수단', '거주지', '회원등급')
    @classmethod
    def no_digit_in_text(cls, v, info):
        v = v.strip()
        if not v:
            raise ValueError(f"'{info.field_name}' 항목은 비워둘 수 없습니다.")
        if any(c.isdigit() for c in v):
            raise ValueError(f"'{info.field_name}' 항목에 숫자를 포함할 수 없습니다.")
        return v

    @field_validator('만족도', '나이', mode='before')
    @classmethod
    def must_be_positive_int(cls, v, info):
        try:
            v = int(v)
        except (ValueError, TypeError):
            raise ValueError(f"'{info.field_name}' 항목은 정수여야 합니다.")
        if info.field_name == '만족도':
            if not (1 <= v <= 5):
                raise ValueError("만족도는 1~5 사이의 값만 입력 가능합니다.")
        elif v < 0:
            raise ValueError(f"'{info.field_name}' 항목은 0 이상이어야 합니다.")
        return v

    @field_validator('선호온도', mode='before')
    @classmethod
    def must_be_float(cls, v):
        try:
            return float(v)
        except (ValueError, TypeError):
            raise ValueError("'선호온도' 항목은 숫자여야 합니다.")


# ════════════════════════════════════════
# 3. DB 초기화
# ════════════════════════════════════════
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            고객ID       TEXT NOT NULL UNIQUE,
            성별         TEXT,
            결제수단     TEXT,
            거주지       TEXT,
            회원등급     TEXT,
            만족도       INTEGER,
            최근접속시간  INTEGER,
            선호온도     REAL,
            나이         INTEGER,
            구매수량     INTEGER,
            총결제금액   INTEGER
        )
    """)
    cursor.execute("SELECT COUNT(*) FROM customers")
    if cursor.fetchone()[0] == 0:
        df = pd.read_csv(CSV_URL).head(10)
        df = df.rename(columns={
            '최근접속시간(시)': '최근접속시간',
            '선호제품군_적정온도': '선호온도'
        })
        data = [
            (row['고객ID'], row['성별'], row['결제수단'], row['거주지'],
             row['회원등급'], int(row['만족도']), int(row['최근접속시간']),
             float(row['선호온도']), int(row['나이']),
             int(row['구매수량']), int(row['총결제금액']))
            for _, row in df.iterrows()
        ]
        cursor.executemany("""
            INSERT INTO customers
            (고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 최근접속시간, 선호온도, 나이, 구매수량, 총결제금액)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, data)
        print(f"✅ CSV 초기 데이터 {len(data)}개 로드 완료")
    conn.commit()
    conn.close()

init_db()


# ════════════════════════════════════════
# 4. 헬퍼 함수
# ════════════════════════════════════════
def get_conn():
    return sqlite3.connect(DB_NAME)

DISP_COLS = "id, 고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이"

def read_df() -> pd.DataFrame:
    """표시용 컬럼만 조회 (구매수량·접속시간·결제금액 제외)"""
    conn = get_conn()
    df = pd.read_sql_query(
        f"SELECT {DISP_COLS} FROM customers ORDER BY id ASC", conn
    )
    conn.close()
    return df

def next_customer_id() -> str:
    """현재 최대 번호 + 1로 다음 고객ID 자동 생성 (예: CUST_0012)"""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT 고객ID FROM customers ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return "CUST_0001"
    match = re.search(r'\d+', row[0])
    next_num = int(match.group()) + 1 if match else 1
    return f"CUST_{next_num:04d}"

def get_summary() -> str:
    """대시보드 상단 요약 통계"""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*), AVG(만족도), AVG(나이) FROM customers")
    row = cursor.fetchone()
    conn.close()
    total, avg_sat, avg_age = row
    return (f"👥 총 고객: **{total}명**  |  "
            f"⭐ 평균 만족도: **{avg_sat:.1f}**  |  "
            f"🎂 평균 나이: **{avg_age:.1f}세**")


# ════════════════════════════════════════
# 5. CRUD 함수
# ════════════════════════════════════════
def db_create(성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이):
    try:
        d = CustomerInput(성별=성별, 결제수단=결제수단, 거주지=거주지,
                          회원등급=회원등급, 만족도=만족도, 선호온도=선호온도, 나이=나이)
    except ValidationError as e:
        msg = e.errors()[0]['msg'].replace('Value error, ', '')
        return f"❌ {msg}", read_df(), get_summary(), next_customer_id()

    new_id = next_customer_id()
    conn = get_conn()
    try:
        conn.execute("""
            INSERT INTO customers (고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (new_id, d.성별, d.결제수단, d.거주지, d.회원등급, d.만족도, d.선호온도, d.나이))
        conn.commit()
        return f"✅ 추가 완료: {new_id}", read_df(), get_summary(), next_customer_id()
    except sqlite3.IntegrityError:
        return f"❌ 중복 오류가 발생했습니다.", read_df(), get_summary(), next_customer_id()
    finally:
        conn.close()


def db_read(search: str):
    conn = get_conn()
    if search.strip():
        df = pd.read_sql_query(
            f"SELECT {DISP_COLS} FROM customers "
            f"WHERE 고객ID LIKE ? OR 거주지 LIKE ? OR 회원등급 LIKE ? OR 성별 LIKE ? "
            f"ORDER BY id ASC",
            conn, params=(f"%{search}%",) * 4
        )
        msg = f"🔍 '{search}' 검색 결과: {len(df)}건"
    else:
        df = pd.read_sql_query(
            f"SELECT {DISP_COLS} FROM customers ORDER BY id ASC", conn
        )
        msg = f"📋 전체 조회: {len(df)}건"
    conn.close()
    return msg, df


def load_customer(고객ID: str):
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요.", "", "", "", "", 0, 0.0, 0
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이 "
        "FROM customers WHERE 고객ID=?", (고객ID,)
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", "", "", "", "", 0, 0.0, 0
    return (f"✅ '{고객ID}' 불러오기 완료", row[0], row[1], row[2], row[3], row[4], row[5], row[6])


def db_update(고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이):
    try:
        d = CustomerInput(성별=성별, 결제수단=결제수단, 거주지=거주지,
                          회원등급=회원등급, 만족도=만족도, 선호온도=선호온도, 나이=나이)
    except ValidationError as e:
        msg = e.errors()[0]['msg'].replace('Value error, ', '')
        return f"❌ {msg}", read_df(), get_summary()

    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE customers SET 성별=?, 결제수단=?, 거주지=?, 회원등급=?, 만족도=?, 선호온도=?, 나이=?
        WHERE 고객ID=?
    """, (d.성별, d.결제수단, d.거주지, d.회원등급, d.만족도, d.선호온도, d.나이, 고객ID))
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    if affected == 0:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", read_df(), get_summary()
    return f"✅ 수정 완료: {고객ID}", read_df(), get_summary()


def db_delete(고객ID: str):
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요.", read_df(), get_summary()
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM customers WHERE 고객ID=?", (고객ID,))
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    if affected == 0:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", read_df(), get_summary()
    return f"✅ 삭제 완료: {고객ID}", read_df(), get_summary()


def greet_user(고객ID: str) -> str:
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요."
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 성별, 거주지, 회원등급, 나이, 만족도 FROM customers WHERE 고객ID=?", (고객ID,)
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"❌ '{고객ID}'를 찾을 수 없습니다."
    return (f"👋 Hello, {고객ID}!\n\n"
            f"  성별     : {row[0]}\n"
            f"  거주지   : {row[1]}\n"
            f"  회원등급 : {row[2]}\n"
            f"  나이     : {row[3]}세\n"
            f"  만족도   : {'⭐' * row[4]} ({row[4]}/5)")


# ════════════════════════════════════════
# 6. 선택 옵션 상수
# ════════════════════════════════════════
OPT_GENDER  = ["남성", "여성"]
OPT_GRADE   = ["Bronze", "Silver", "Gold", "VIP", "VVIP"]
OPT_PAYMENT = ["신용카드", "계좌이체", "휴대폰결제", "간편결제"]
OPT_REGION  = ["서울", "부산", "대구", "광주", "인천", "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]


# ════════════════════════════════════════
# 7. Gradio UI
# ════════════════════════════════════════
css = """
.tab-nav button { font-size: 15px !important; font-weight: 600 !important; }
.status-box textarea { font-size: 14px !important; }
.summary-box { font-size: 15px !important; }

/* textarea 크기조절 핸들(우하단 화살표) 제거 */
textarea { resize: none !important; }
"""

with gr.Blocks(title="Customer CRUD Dashboard", theme=gr.themes.Soft(), css=css) as demo:

    # ── 헤더 ──
    gr.Markdown("# 🗄️ Customer CRUD Dashboard")
    gr.Markdown("---")
    summary_md = gr.Markdown(value=get_summary(), elem_classes="summary-box")
    gr.Markdown("---")

    with gr.Tabs():

        # ════════ ➕ Create ════════
        with gr.Tab("➕  Create"):
            gr.Markdown("### 새 고객 등록")
            gr.Markdown("> 고객ID는 **자동 배정**됩니다. 드롭다운에서 선택하거나 직접 입력하세요.")

            with gr.Row():
                auto_id = gr.Textbox(
                    label="자동 배정될 고객ID",
                    value=next_customer_id,
                    interactive=False,
                    scale=1
                )

            gr.Markdown("**기본 정보**")
            with gr.Row():
                c_gender  = gr.Dropdown(label="성별",     choices=OPT_GENDER,  allow_custom_value=True, scale=1)
                c_grade   = gr.Dropdown(label="회원등급", choices=OPT_GRADE,   allow_custom_value=True, scale=1)
                c_payment = gr.Dropdown(label="결제수단", choices=OPT_PAYMENT, allow_custom_value=True, scale=2)
                c_region  = gr.Dropdown(label="거주지",   choices=OPT_REGION,  allow_custom_value=True, scale=2)

            gr.Markdown("**수치 정보**")
            with gr.Row():
                c_age  = gr.Number(label="나이",          value=0, minimum=0,  maximum=120, step=1,   precision=0, scale=1)
                c_sat  = gr.Number(label="만족도 (1~5)",  value=0, step=1, precision=0, scale=1)
                c_temp = gr.Number(label="선호온도 (℃)", value=0, minimum=-10, maximum=50,  step=0.5,              scale=1)

            with gr.Row():
                c_btn = gr.Button("➕  고객 추가", variant="primary", scale=1)

            c_status = gr.Textbox(label="결과", interactive=False, elem_classes="status-box")
            c_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            c_btn.click(
                fn=db_create,
                inputs=[c_gender, c_payment, c_region, c_grade, c_sat, c_temp, c_age],
                outputs=[c_status, c_table, summary_md, auto_id]
            )

        # ════════ 🔍 Read ════════
        with gr.Tab("🔍  Read"):
            gr.Markdown("### 고객 조회")
            gr.Markdown("💡 **검색 가능 항목**: 고객ID · 거주지 · 회원등급 · 성별  |  비워두면 전체 조회됩니다.")
            with gr.Row():
                r_search = gr.Textbox(
                    label="검색",
                    placeholder="예) CUST_0001  /  서울  /  Gold  /  남성",
                    scale=4
                )
                r_btn = gr.Button("🔍  조회", variant="primary", scale=1)
            r_status = gr.Textbox(label="결과", interactive=False, elem_classes="status-box")
            r_table  = gr.DataFrame(label="조회 결과", value=read_df)

            r_btn.click(fn=db_read, inputs=[r_search], outputs=[r_status, r_table])

        # ════════ ✏️ Update ════════
        with gr.Tab("✏️  Update"):
            gr.Markdown("### 고객 정보 수정")
            gr.Markdown("> **① 고객ID 입력 → 불러오기** &nbsp;→&nbsp; **② 값 수정** &nbsp;→&nbsp; **③ 수정하기**")

            with gr.Row():
                u_id       = gr.Textbox(label="고객ID", placeholder="예) CUST_0001", scale=4)
                u_load_btn = gr.Button("📂  불러오기", variant="secondary", scale=1)
            u_load_status = gr.Textbox(label="불러오기 결과", interactive=False, elem_classes="status-box")

            gr.Markdown("**기본 정보**")
            with gr.Row():
                u_gender  = gr.Dropdown(label="성별",     choices=OPT_GENDER,  allow_custom_value=True, scale=1)
                u_grade   = gr.Dropdown(label="회원등급", choices=OPT_GRADE,   allow_custom_value=True, scale=1)
                u_payment = gr.Dropdown(label="결제수단", choices=OPT_PAYMENT, allow_custom_value=True, scale=2)
                u_region  = gr.Dropdown(label="거주지",   choices=OPT_REGION,  allow_custom_value=True, scale=2)

            gr.Markdown("**수치 정보**")
            with gr.Row():
                u_age  = gr.Number(label="나이",          value=0, minimum=0,   maximum=120, step=1,   precision=0, scale=1)
                u_sat  = gr.Number(label="만족도 (1~5)",  value=0, step=1, precision=0, scale=1)
                u_temp = gr.Number(label="선호온도 (℃)", value=0, minimum=-10,  maximum=50,  step=0.5,              scale=1)

            with gr.Row():
                u_btn = gr.Button("✏️  수정하기", variant="primary", scale=1)

            u_status = gr.Textbox(label="결과", interactive=False, elem_classes="status-box")
            u_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            u_load_btn.click(
                fn=load_customer,
                inputs=[u_id],
                outputs=[u_load_status, u_gender, u_payment, u_region, u_grade, u_sat, u_temp, u_age]
            )
            u_btn.click(
                fn=db_update,
                inputs=[u_id, u_gender, u_payment, u_region, u_grade, u_sat, u_temp, u_age],
                outputs=[u_status, u_table, summary_md]
            )

        # ════════ 👋 Greet ════════
        with gr.Tab("👋  Greet"):
            gr.Markdown("### 고객 인사말")
            gr.Markdown("> 고객ID를 입력하면 해당 고객 정보와 함께 인사말을 출력합니다.")
            with gr.Row():
                g_id  = gr.Textbox(label="고객ID", placeholder="예) CUST_0001", scale=4)
                g_btn = gr.Button("👋  인사하기", variant="primary", scale=1)
            g_output = gr.Textbox(label="결과", lines=7, interactive=False, elem_classes="status-box")

            g_btn.click(fn=greet_user, inputs=[g_id], outputs=[g_output])

        # ════════ 🗑️ Delete ════════
        with gr.Tab("🗑️  Delete"):
            gr.Markdown("### 고객 삭제")
            gr.Markdown("> ⚠️ 삭제된 데이터는 복구되지 않습니다.")
            with gr.Row():
                d_id  = gr.Textbox(label="삭제할 고객ID", placeholder="예) CUST_0003", scale=4)
                d_btn = gr.Button("🗑️  삭제하기", variant="stop", scale=1)
            d_status = gr.Textbox(label="결과", interactive=False, elem_classes="status-box")
            d_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            d_btn.click(fn=db_delete, inputs=[d_id], outputs=[d_status, d_table, summary_md])

demo.launch()

✅ CSV 초기 데이터 10개 로드 완료


/tmp/ipykernel_3651/177066697.py:306: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Customer CRUD Dashboard", theme=gr.themes.Soft(), css=css) as demo:
/tmp/ipykernel_3651/177066697.py:306: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Customer CRUD Dashboard", theme=gr.themes.Soft(), css=css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cea2e74629d859f86d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
